# 3rd EDA · STEP 2 — 상관분석 (보고서 3종 발행 기업)**입력** `ESG_3종발행_분석데이터_정규화.csv` — STEP 1의 출력**표본** 105개 기업-연도 / 37개 기업 / 2022~2025 / 전부 KOSPI---### STEP 1에서 넘어온 조건| 항목 | 상태 ||---|---|| 공시량 편향 | **통제됨** — 3종 모두 발행한 기업만 남겨 입력 조건이 동일 || 범위 제한 | **발생** — 등급 SD가 0.75~0.91배로 축소, 상관이 구조적으로 낮게 나옴 || 업종 효과 | **유의** (p=0.0005~0.037) — 표준화하지 않고 이 노트북에서 통제변수로 다룸 || 표준화 | Z-score(`*_z`) · 백분위(`*_pct`) 사용 가능, 상관 자체는 원점수와 동일 || 이상치 | 제거하지 않음 (측정 오류 아님) → 순위 기반 Spearman을 기본으로 |### 이 노트북이 답하려는 것| # | 질문 ||---|---|| 1 | 공시 교란을 걷어낸 뒤에도 등급-점수 상관이 남는가 || 2 | 영역별로 얼마나 다른가 — E는 정말 유의하게 약한가 || 3 | 인접 등급을 갈라내는가 || 4 | **E/S/G를 구분해서** 재고 있는가 (판별타당도) || 5 | 상관이 업종 구성 때문에 생긴 것은 아닌가 || 6 | 범위 제한을 보정하면 실제 수준은 어느 정도인가 || 7 | 같은 기업의 연도별 변화를 따라가는가 || 8 | 어느 기업에서 가장 크게 어긋나는가 |

## 0. 환경 설정

In [ ]:
# 한글 폰트 (코랩 최초 1회, 약 30초). 런타임 재시작 불필요.
!apt-get -qq install -y fonts-nanum > /dev/null
!fc-cache -f > /dev/null

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm._load_fontmanager(try_read_cache=False)
matplotlib.rc("font", family="NanumGothic")
matplotlib.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
print("폰트:", matplotlib.rcParams["font.family"])

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. 데이터 로드STEP 1이 만든 `ESG_3종발행_분석데이터_정규화.csv` 를 올린다.(같은 세션에서 STEP 1을 먼저 돌렸다면 이미 `/content` 에 있으므로 `USE_UPLOAD = False`)

In [ ]:
USE_UPLOAD = True
CSV = "ESG_3종발행_분석데이터_정규화.csv"

if USE_UPLOAD:
    from google.colab import files
    CSV = list(files.upload().keys())[0]

d = pd.read_csv(CSV, dtype={"기업코드": str})
print("shape:", d.shape)
print(f"{len(d)}개 기업-연도 / {d['기업코드'].nunique()}개 기업 / "
      f"{sorted(d['평가년도'].unique())} / {d['시장구분'].unique()}")
d.head(3)

In [ ]:
# ── 상수 ─────────────────────────────────────────────────
ORD = {"D": 1, "C": 2, "B": 3, "B+": 4, "A": 5, "A+": 6, "S": 7}
LAB = {v: k for k, v in ORD.items()}
GCOLS = ["ESG등급", "환경", "사회", "지배구조"]
SCOLS = ["전체결과", "P결과", "E결과", "S결과", "G결과"]
ECOLS = ["E결과", "S결과", "G결과"]
PAIRS = [("전체", "ESG등급", "전체결과"), ("환경(E)", "환경", "E결과"),
         ("사회(S)", "사회", "S결과"), ("지배구조(G)", "지배구조", "G결과")]

# 비교 기준 — STEP 1 이전 전체 표본(171행, 공시 조건 혼재)에서의 값
FULL = {"전체": 0.679, "환경(E)": 0.623, "사회(S)": 0.676, "지배구조(G)": 0.658}
FULL_GRADE_SD = {"ESG등급": 1.48, "환경": 1.33, "사회": 1.66, "지배구조": 1.45}

d["업종4"] = d["산업코드"].astype(str).str[:4]
print("등급 분포")
display(pd.DataFrame({g: d[g].value_counts() for g in GCOLS})
        .reindex(list(ORD)[::-1]).fillna(0).astype(int))

## 2. 등급 ↔ 점수 상관

In [ ]:
rows = []
for name, g, s in PAIRS:
    x, y = d[g + "_ord"], d[s]
    n = len(d)
    pr, sp, kt = stats.pearsonr(x, y), stats.spearmanr(x, y), stats.kendalltau(x, y)
    z, se = np.arctanh(sp.statistic), 1 / np.sqrt(n - 3)     # Fisher z 신뢰구간
    rows.append({"영역": name, "n": n,
                 "Pearson r": pr.statistic, "Spearman ρ": sp.statistic,
                 "95% CI 하한": np.tanh(z - 1.96 * se), "95% CI 상한": np.tanh(z + 1.96 * se),
                 "Kendall τ": kt.statistic, "p": sp.pvalue,
                 "전체표본 ρ": FULL[name], "변화": sp.statistic - FULL[name]})
corr = pd.DataFrame(rows).set_index("영역")
display(corr.round(4))

> **공시 교란을 걷어내도 상관은 남는다.** 네 영역 모두 p<1e-6이고 95% 신뢰구간이 0을 포함하지 않는다.>> 다만 **E만 크게 떨어졌다** (0.623 → 0.480). 전체·S·G는 0.02~0.08 하락에 그친다.> 즉 이전 EDA에서 E가 보이던 상관의 상당 부분은 "지속가능경영보고서를 냈는가"가 만든 것이었다.

In [ ]:
# 반복측정 보정 — 같은 기업이 최대 4개 연도 들어 있으므로 기업 단위로 재표집한다.
# (약 40초 소요)
rng = np.random.default_rng(42)
groups = {c: g.index.to_numpy() for c, g in d.groupby("기업코드")}
codes = np.array(list(groups))
B = 2000

rows = []
for name, g, s in PAIRS:
    x, y = d[g + "_ord"].to_numpy(float), d[s].to_numpy(float)
    vals = np.empty(B)
    for b in range(B):
        pick = rng.choice(codes, len(codes), replace=True)
        ii = np.concatenate([groups[c] for c in pick])
        vals[b] = stats.spearmanr(x[ii], y[ii]).statistic
    lo, hi = np.percentile(vals, [2.5, 97.5])
    rows.append({"영역": name, "Spearman ρ": stats.spearmanr(x, y).statistic,
                 "클러스터 CI 하한": lo, "클러스터 CI 상한": hi, "CI 폭": hi - lo})
print("■ 기업 클러스터 부트스트랩 95% CI (B=2000)")
display(pd.DataFrame(rows).set_index("영역").round(3))
print("※ 단순 Fisher CI보다 넓다. 같은 기업의 연도 관측이 독립이 아니기 때문이다.")

In [ ]:
# 기업 단위(연도 평균) — 연도 반복을 아예 없앤 보수적 관점
agg = (d.groupby(["기업코드", "기업명"])
         .agg({**{s: "mean" for _, _, s in PAIRS},
               **{g + "_ord": "mean" for _, g, _ in PAIRS}})
         .reset_index())
print(f"■ 기업 단위 상관 (n={len(agg)})")
display(pd.DataFrame([{
    "영역": name,
    "Spearman ρ": stats.spearmanr(agg[g + "_ord"], agg[s]).statistic,
    "p": stats.spearmanr(agg[g + "_ord"], agg[s]).pvalue,
} for name, g, s in PAIRS]).set_index("영역").round(4))

> 기업 단위로 집계하면 전체 0.757 · S 0.778 · G 0.730 으로 올라가는데 **E만 0.532** 에 머문다.> 관측 단위를 바꿔도 E의 열세는 그대로다.

## 3. 순서일치도 — 등급이 다른 두 기업의 점수 순서가 맞는가

In [ ]:
# 등급이 다른 모든 쌍에 대해 점수 순서가 등급 순서와 같은지 센다 (Somers' D 계열)
def order_agreement(gv, sv):
    a, b = np.asarray(gv), np.asarray(sv)
    conc = disc = tie = 0
    for i, j in itertools.combinations(range(len(a)), 2):
        if a[i] == a[j]:
            continue
        gs, ss = np.sign(a[i] - a[j]), np.sign(b[i] - b[j])
        if ss == 0:
            tie += 1
        elif gs == ss:
            conc += 1
        else:
            disc += 1
    tot = conc + disc + tie
    return conc / tot, disc / tot, tie / tot, (conc + .5 * tie) / tot, tot


rows = []
for name, g, s in PAIRS:
    c, ds, t, agr, tot = order_agreement(d[g + "_ord"], d[s])
    rows.append({"영역": name, "일치": c, "역전": ds, "동점": t, "순서일치도": agr, "비교 쌍": tot})
display(pd.DataFrame(rows).set_index("영역").round(4))

## 4. 등급별 평균과 단조성

In [ ]:
for name, g, s in PAIRS:
    t = d.groupby(g + "_ord")[s].agg(["count", "mean", "std", "min", "max"]).round(1)
    t.index = [LAB[i] for i in t.index]
    m = d.groupby(g + "_ord")[s].mean()
    inv = [f"{LAB[m.index[k]]}→{LAB[m.index[k+1]]}" for k in range(len(m) - 1)
           if m.iloc[k + 1] < m.iloc[k]]
    print(f"[{name}] 역전 구간 {len(inv)}/{len(m)-1}" + (f"  → {', '.join(inv)}" if inv else ""))
    display(t)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, g, s) in zip(axes, PAIRS):
    lv = sorted(d[g + "_ord"].unique())
    ax.boxplot([d.loc[d[g + "_ord"] == l, s] for l in lv], showmeans=True)
    ax.set_xticklabels([f"{LAB[l]}\n(n={int((d[g+'_ord']==l).sum())})" for l in lv], fontsize=8)
    m = d.groupby(g + "_ord")[s].mean()
    ax.plot(range(1, len(lv) + 1), m.values, "o-", color="crimson", lw=1.3, ms=4)
    ax.set_title(name); ax.grid(axis="y", alpha=.3)
axes[0].set_ylabel("채점 점수")
plt.suptitle("등급별 점수 분포 — 붉은 선이 평균, 꺾이면 단조성 위반", y=1.03)
plt.tight_layout(); plt.show()

## 5. 인접 등급 판별력

In [ ]:
rows = []
for name, g, s in PAIRS:
    lv = sorted(d[g + "_ord"].unique())
    for a, b in zip(lv[:-1], lv[1:]):
        x, y = d.loc[d[g + "_ord"] == a, s], d.loc[d[g + "_ord"] == b, s]
        if len(x) < 3 or len(y) < 3:
            rows.append({"영역": name, "구간": f"{LAB[a]}→{LAB[b]}", "n(하)": len(x), "n(상)": len(y),
                         "Cohen d": np.nan, "AUC": np.nan, "판정": "표본 부족"})
            continue
        pooled = np.sqrt(((len(x) - 1) * x.var() + (len(y) - 1) * y.var()) / (len(x) + len(y) - 2))
        auc = stats.mannwhitneyu(y, x).statistic / (len(x) * len(y))
        rows.append({"영역": name, "구간": f"{LAB[a]}→{LAB[b]}", "n(하)": len(x), "n(상)": len(y),
                     "Cohen d": (y.mean() - x.mean()) / pooled, "AUC": auc,
                     "판정": ("역방향" if auc < .45 else "구분 못함" if auc < .6
                             else "약함" if auc < .7 else "구분됨")})
adj = pd.DataFrame(rows)
display(adj.round(2).style.map(
    lambda v: {"역방향": "background-color:#f8cbad", "구분 못함": "background-color:#fce4d6",
               "구분됨": "background-color:#e2efda"}.get(v, ""), subset=["판정"]))

> **상위 등급대는 갈라내고, 하위 등급대는 오히려 뒤집힌다.**>> - `A→A+` 는 네 영역 모두 AUC 0.75~0.81 로 잘 구분한다.> - 반면 E의 `C→B`(AUC 0.33)와 S의 `C→B`(AUC 0.14)는 **등급이 오르는데 점수가 내려간다.**> - 다만 이 구간은 n이 4~11로 작다. 방향은 신뢰하되 크기는 확정적으로 읽지 않는다.> - 3종 발행 표본은 D·C 등급이 거의 없어(전체 D 2건) 하위 판별력은 원래 이 표본으로 평가하기 어렵다.

## 6. 판별타당도 — E/S/G를 구분해서 재고 있는가영역 점수는 **자기 영역 등급**과 가장 강하게 붙어야 한다.대각선이 서지 않으면 세 영역이 같은 것을 재고 있다는 뜻이다.

In [ ]:
cross = pd.DataFrame({g: [stats.spearmanr(d[s], d[g + "_ord"]).statistic for s in SCOLS]
                      for g in GCOLS}, index=SCOLS)
display(cross.round(3).style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))

rows = []
for s, g in [("E결과", "환경"), ("S결과", "사회"), ("G결과", "지배구조")]:
    own = cross.loc[s, g]
    oth = cross.loc[s, [c for c in GCOLS if c not in (g, "ESG등급")]]
    rows.append({"점수": s, "자기 등급": g, "ρ(자기)": own,
                 "최대 타영역": oth.idxmax(), "ρ(타영역)": oth.max(),
                 "차이": own - oth.max(),
                 "판정": "자기 영역 우위" if own - oth.max() > .03 else "구분 실패"})
print("■ 자기 영역 우위")
display(pd.DataFrame(rows).set_index("점수").round(3))

print("■ 점수 간 상관 (높을수록 단일 축)")
display(d[SCOLS].corr(method="spearman").round(3)
        .style.background_gradient(cmap="Oranges", axis=None).format("{:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
im = axes[0].imshow(cross.values, cmap="Blues", vmin=.35, vmax=.70)
axes[0].set_xticks(range(len(GCOLS)), GCOLS)
axes[0].set_yticks(range(len(SCOLS)), SCOLS)
for i in range(len(SCOLS)):
    for j in range(len(GCOLS)):
        axes[0].text(j, i, f"{cross.values[i, j]:.3f}", ha="center", va="center",
                     color="white" if cross.values[i, j] > .58 else "black", fontsize=9)
axes[0].set_title("점수 × 등급 교차 상관 (Spearman ρ)")
plt.colorbar(im, ax=axes[0])

lbl = ["E결과", "S결과", "G결과"]
own = [cross.loc["E결과", "환경"], cross.loc["S결과", "사회"], cross.loc["G결과", "지배구조"]]
oth = [max(cross.loc["E결과", ["사회", "지배구조"]]),
       max(cross.loc["S결과", ["환경", "지배구조"]]),
       max(cross.loc["G결과", ["환경", "사회"]])]
xx = np.arange(3)
axes[1].bar(xx - .2, own, .4, label="자기 영역 등급")
axes[1].bar(xx + .2, oth, .4, label="타 영역 등급 (최대)")
axes[1].set_xticks(xx, lbl); axes[1].legend(); axes[1].grid(axis="y", alpha=.3)
axes[1].set_ylabel("Spearman ρ"); axes[1].set_title("자기 영역이 더 높아야 정상")
plt.tight_layout(); plt.show()

> **G는 확보, S는 소폭, E는 여전히 실패.**>> | 점수 | 자기 등급 | 최대 타영역 | 차이 | 전체표본에서는 |> |---|---:|---:|---:|---:|> | E결과 | 0.480 | 0.486 (사회) | **−0.006** | −0.010 |> | S결과 | 0.649 | 0.614 (환경) | +0.035 | −0.000 |> | G결과 | 0.577 | 0.481 (환경) | **+0.096** | +0.062 |>> 공시 조건을 맞추자 S와 G는 자기 영역 우위를 얻었지만, **E는 여전히 사회등급과 더 잘 붙는다.**> 점수 간 상관도 E↔S가 0.774로 (전체표본 0.877에서 내려왔지만) 여전히 높다.

## 7. 영역 간 상관 차이가 통계적으로 유의한가

In [ ]:
# 같은 표본에서 계산한 상관들의 차이 — 부트스트랩으로 CI를 잡는다 (약 30초)
rng2 = np.random.default_rng(7)
B2 = 2000
idx = rng2.integers(0, len(d), (B2, len(d)))
boot = {}
for name, g, s in PAIRS:
    x, y = d[g + "_ord"].to_numpy(float), d[s].to_numpy(float)
    boot[name] = np.array([stats.spearmanr(x[i], y[i]).statistic for i in idx])

rows = []
for a, b in [("환경(E)", "사회(S)"), ("환경(E)", "지배구조(G)"), ("환경(E)", "전체"),
             ("사회(S)", "지배구조(G)")]:
    diff = boot[a] - boot[b]
    lo, hi = np.percentile(diff, [2.5, 97.5])
    rows.append({"비교": f"{a} − {b}", "차이": diff.mean(), "CI 하한": lo, "CI 상한": hi,
                 "판정": "유의하게 다름" if lo * hi > 0 else "구분 안 됨"})
display(pd.DataFrame(rows).set_index("비교").round(3))

> **E는 S보다 유의하게 낮다** (차이 −0.169, CI [−0.323, −0.021]).> E와 G의 차이는 신뢰구간이 0을 걸쳐 확정할 수 없다.> 즉 "E가 가장 약하고 S가 가장 강하다"까지는 말할 수 있고, "E와 G 중 어느 쪽이 더 나쁜가"는 이 표본으로 못 가른다.

## 8. 업종 통제 — 상관이 업종 구성 때문은 아닌가

In [ ]:
vc = d["업종4"].value_counts()
big = vc[vc >= 5].index
sub = d[d["업종4"].isin(big)].copy()
print(f"업종 분포 (n>=5 인 {len(big)}개 업종 / {len(sub)}행 사용)")
display(vc.rename("행 수").to_frame().T)

rows = []
for name, g, s in PAIRS:
    ss = sub.copy()
    ss["gd"] = ss[g + "_ord"] - ss.groupby("업종4")[g + "_ord"].transform("mean")
    ss["sd"] = ss[s] - ss.groupby("업종4")[s].transform("mean")
    w = stats.pearsonr(ss["gd"], ss["sd"])
    rows.append({"영역": name,
                 "전체 ρ": stats.spearmanr(d[g + "_ord"], d[s]).statistic,
                 "업종 내 편차 r": w.statistic, "p": w.pvalue,
                 "변화": w.statistic - stats.spearmanr(d[g + "_ord"], d[s]).statistic})
print("■ 업종 평균을 뺀 뒤에도 상관이 남는가")
display(pd.DataFrame(rows).set_index("영역").round(4))

In [ ]:
# 회귀로 다시 확인 — 등급 vs 업종더미
y = d["전체결과"].to_numpy(float)
dum = pd.get_dummies(d["업종4"], drop_first=True).astype(float).to_numpy()


def ols(X):
    X1 = np.column_stack([np.ones(len(X))] + [X[:, i] for i in range(X.shape[1])])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    r2 = 1 - ((y - X1 @ b) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    adj = 1 - (1 - r2) * (len(y) - 1) / (len(y) - X1.shape[1])
    return b, r2, adj


models = [("등급만", d[["ESG등급_ord"]].to_numpy(float)),
          ("업종더미만", dum),
          ("등급 + 업종더미", np.column_stack([d["ESG등급_ord"].to_numpy(float), dum]))]
rows = []
for label, X in models:
    b, r2, adj = ols(X)
    rows.append({"모형": label, "변수 수": X.shape[1], "R²": r2, "adj R²": adj,
                 "등급 계수": b[1] if label != "업종더미만" else np.nan})
display(pd.DataFrame(rows).set_index("모형").round(3))

> **상관은 업종 구성의 산물이 아니다.** 업종 평균을 뺀 편차끼리 봐도 전체 0.631 · S 0.614 · G 0.615 가> 그대로 남고(모두 p<0.001), G는 오히려 올라간다(0.577 → 0.615).>> 회귀에서도 등급만으로 R²=0.432 인데 업종더미만으로는 0.191 이다.> 이전 EDA에서 공시량(R²=0.510)이 등급(0.425)을 앞질렀던 것과 대조적이다 — **이제는 등급이 주된 설명변수다.**

## 9. 범위 제한 보정

In [ ]:
# Thorndike Case II — 3종 필터로 좁아진 등급 산포를 전체 표본 기준으로 되돌린 참고치
rows = []
for name, g, s in PAIRS:
    r = stats.pearsonr(d[g + "_ord"], d[s]).statistic
    k = FULL_GRADE_SD[g] / d[g + "_ord"].std(ddof=1)
    rows.append({"영역": name, "전체표본 등급SD": FULL_GRADE_SD[g],
                 "3종 등급SD": d[g + "_ord"].std(ddof=1), "축소비 k": k,
                 "관측 r": r, "보정 r": r * k / np.sqrt(1 + r ** 2 * (k ** 2 - 1))})
display(pd.DataFrame(rows).set_index("영역").round(3))
print("※ 선택이 제3변수(공시 조건)에 걸려 있어 근사치다. 절대값이 아니라 과소평가 폭의 감으로만 쓴다.")

> 보정하면 전체 0.72 · S 0.72 · G 0.64 · **E 0.56** 수준이다.> 범위 제한을 감안해도 **E만 눈에 띄게 낮다**는 결론은 바뀌지 않는다.

## 10. 기업 내 변화 추적

In [ ]:
multi = d[d.groupby("기업코드")["기업코드"].transform("size") >= 2]
print(f"2개 연도 이상 보유: {multi['기업코드'].nunique()}개 기업 / {len(multi)}행")

rows = []
for name, g, s in PAIRS:
    dd = multi[[g + "_ord", s, "기업코드"]].copy()
    dd["gd"] = dd[g + "_ord"] - dd.groupby("기업코드")[g + "_ord"].transform("mean")
    dd["sd"] = dd[s] - dd.groupby("기업코드")[s].transform("mean")
    wr = stats.pearsonr(dd["gd"], dd["sd"])                       # 기업 고정효과 제거

    e = d.sort_values(["기업코드", "평가년도"]).copy()               # 1차 차분
    e["dg"] = e.groupby("기업코드")[g + "_ord"].diff()
    e["ds"] = e.groupby("기업코드")[s].diff()
    ee = e.dropna(subset=["dg", "ds"])
    mv = ee[ee.dg != 0]
    same = np.sign(mv.dg) == np.sign(mv.ds)
    dr = stats.pearsonr(ee["dg"], ee["ds"])

    rows.append({"영역": name, "기업내 r": wr.statistic, "p": wr.pvalue,
                 "Δ상관 r": dr.statistic, "p ": dr.pvalue,
                 "등급 변동": len(mv), "방향 일치": int(same.sum()), "일치율": same.mean()})
display(pd.DataFrame(rows).set_index("영역").round(3))

> **개선은 있으나 여전히 약하다.**>> 전체 기준 방향 일치율이 **63.2% → 75.9%** 로 올랐고 기업내 r도 유의해졌다(0.294, p=0.003).> 하지만 Δ등급 vs Δ점수 상관은 전체 0.229(p=0.060)로 여전히 경계선이고, E·S·G는 유의하지 않다.> 등급 변동 표본이 26~33건뿐이라 검정력 자체가 부족하다.> **연도별 개선 추적은 아직 이 모델의 용도가 아니다.**

## 11. 불일치 사례

In [ ]:
dd = d.copy()
b1, a1 = np.polyfit(dd["ESG등급_ord"], dd["전체결과"], 1)
dd["기대점수"] = b1 * dd["ESG등급_ord"] + a1
dd["잔차"] = dd["전체결과"] - dd["기대점수"]
dd["z"] = dd["잔차"] / dd["잔차"].std()
cols = ["기업명", "평가년도", "ESG등급", "전체결과", "기대점수", "잔차", "z", "업종4"]

print("■ 모델이 과대평가 (등급 대비 점수 높음) 상위 10")
display(dd.nlargest(10, "z")[cols].round(1))
print("■ 모델이 과소평가 하위 10")
display(dd.nsmallest(10, "z")[cols].round(1))

In [ ]:
agg2 = agg.copy()
agg2["점수순위"] = agg2["전체결과"].rank(ascending=False)
agg2["등급순위"] = agg2["ESG등급_ord"].rank(ascending=False)
agg2["순위차"] = agg2["점수순위"] - agg2["등급순위"]

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.scatter(agg2["등급순위"], agg2["점수순위"], s=32, alpha=.7)
ax.plot([0, len(agg2)], [0, len(agg2)], "--", color="gray", lw=1)
for _, r in pd.concat([agg2.nlargest(5, "순위차"), agg2.nsmallest(5, "순위차")]).iterrows():
    ax.annotate(r["기업명"], (r["등급순위"], r["점수순위"]), fontsize=8,
                xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("KCGS 등급 순위 (1위 = 최상위)"); ax.set_ylabel("채점 점수 순위")
ax.set_title(f"기업 단위 순위 비교 (n={len(agg2)})\n대각선 위 = 모델이 과소평가, 아래 = 과대평가")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

display(agg2.sort_values("순위차")[["기업명", "ESG등급_ord", "전체결과",
                                   "등급순위", "점수순위", "순위차"]].round(2))

## 12. 결론### 표본보고서 3종을 모두 발행한 **105개 기업-연도 / 37개 기업 / 전부 KOSPI**.공시량 편향이 설계 단계에서 통제되었고, 그 대가로 범위 제한이 생겼다.### 확인된 것**1. 공시 교란을 걷어내도 상관은 남는다**| 영역 | ρ (3종) | 95% CI | 클러스터 CI | 전체표본 ρ | 순서일치도 ||---|---:|---|---|---:|---:|| 전체 | **0.655** | [0.529, 0.752] | [0.492, 0.774] | 0.679 | 81.3% || 사회(S) | **0.649** | [0.522, 0.748] | [0.469, 0.758] | 0.676 | 81.7% || 지배구조(G) | 0.577 | [0.433, 0.692] | [0.370, 0.735] | 0.658 | 75.1% || **환경(E)** | **0.480** | [0.317, 0.615] | [0.285, 0.629] | 0.623 | 72.7% |기업 단위로 집계하면 전체 0.757 · S 0.778 · G 0.730 · **E 0.532**.**2. E가 유의하게 가장 약하다**E − S = −0.169 (CI [−0.323, −0.021]) 로 유의. E − G 는 신뢰구간이 0을 걸쳐 확정 불가.범위 제한 보정 후에도 E는 0.56 으로 나머지(0.64~0.72)와 벌어진다.**이전 EDA에서 E가 보이던 상관의 상당 부분은 지속가능경영보고서 유무가 만든 것이었다.****3. 판별타당도는 G만 확보, E는 여전히 실패**| 점수 | 자기 등급 | 최대 타영역 | 차이 ||---|---:|---:|---:|| G결과 | 0.577 | 0.481 (환경) | **+0.096** || S결과 | 0.649 | 0.614 (환경) | +0.035 || E결과 | 0.480 | 0.486 (사회) | **−0.006** |E점수는 환경등급보다 **사회등급과 더 잘 붙는다.** E↔S 점수 상관도 0.774로 높다.**4. 상관은 업종 구성의 산물이 아니다**업종 평균을 뺀 편차끼리도 전체 0.631 · S 0.614 · G 0.615 (모두 p<0.001).회귀에서 등급만 R²=0.432 vs 업종더미만 R²=0.191.이전 EDA에서 공시량(0.510)이 등급(0.425)을 앞질렀던 구도가 뒤집혔다.**5. 상위 등급은 갈라내고 하위 등급은 뒤집힌다**`A→A+` AUC 0.75~0.81. 반면 E의 `C→B` 0.33, S의 `C→B` 0.14 로 역방향.단 해당 구간 n이 4~11이라 방향만 신뢰한다.**6. 연도 변화 추적은 아직 안 된다**방향 일치율 63.2% → 75.9% 로 개선됐지만 Δ상관은 전체 0.229(p=0.060)로 경계선,E·S·G는 유의하지 않다.---### 종합**3종 발행 기업으로 좁히자 모델의 성격이 분명해졌다.**공시 조건이 같을 때 이 모델은 기업 간 ESG 서열을 ρ≈0.65 수준으로 재현하며,그 상관은 업종이나 공시량이 아니라 등급 자체와 붙어 있다.문제는 **환경(E)** 하나로 좁혀진다. E는 (a) 상관이 유의하게 가장 낮고,(b) 자기 영역 등급보다 사회 등급과 더 잘 붙으며, (c) 하위 등급 구간에서 순서가 뒤집힌다.1차 검수에서 나온 원인(정량 표 추출 실패 → 채점기준 문장 되풀이 146건)과 방향이 일치한다.### STEP 3 제안| # | 할 일 | 근거 ||---|---|---|| ① | E 항목 32개를 정량/정성으로 나눠 각각 상관 재측정 | E 실패가 정량 지표에 몰려 있는지 확인 || ② | Factbook 표 청크 검색·파싱 개선 후 A/B | E의 실패 원인으로 지목된 지점 || ③ | S·G는 현 상태로 확정하고 기준선 유지 | 판별타당도 확보, ρ 0.58~0.65 || ④ | 하위 등급(C·D) 표본 보강 | 3종 표본에 D가 2건뿐이라 하위 판별력 평가 불가 |

In [ ]:
# 요약 지표를 CSV로 내보내기 (STEP 3 / 보고용)
summary = []
for name, g, s in PAIRS:
    x, y = d[g + "_ord"], d[s]
    sp = stats.spearmanr(x, y)
    z, se = np.arctanh(sp.statistic), 1 / np.sqrt(len(d) - 3)
    _, _, _, agr, _ = order_agreement(x, y)
    r = stats.pearsonr(x, y).statistic
    k = FULL_GRADE_SD[g] / x.std(ddof=1)
    summary.append({
        "영역": name, "n": len(d),
        "Pearson r": round(r, 3), "Spearman ρ": round(sp.statistic, 3),
        "CI 하한": round(np.tanh(z - 1.96 * se), 3), "CI 상한": round(np.tanh(z + 1.96 * se), 3),
        "Kendall τ": round(stats.kendalltau(x, y).statistic, 3),
        "순서일치도": round(agr, 3),
        "범위제한 보정 r": round(r * k / np.sqrt(1 + r ** 2 * (k ** 2 - 1)), 3),
        "전체표본 ρ": FULL[name],
    })
summ = pd.DataFrame(summary).set_index("영역")
summ.to_csv("STEP2_상관요약_3종발행.csv", encoding="utf-8-sig")
display(summ)
print("저장: STEP2_상관요약_3종발행.csv")

# from google.colab import files; files.download("STEP2_상관요약_3종발행.csv")